In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

print("=" * 60)
print("🏗️ STEP 1: GENERATING FIXTURE CONTEXT")
print("=" * 60)

# 1. Define Paths 
predictions_path = Path("../data/processed/world_cup_predictions_final.csv")
fixtures_path = Path("../scrapers/elo_ratings/output/wc2026_fixtures.csv")
output_fixture_path = Path("../data/processed/fixture_context.csv")
output_final_path = Path("../data/processed/world_cup_predictions_adjusted.csv")

# 2. Load Data
df_players = pd.read_csv(predictions_path)
df_fixtures = pd.read_csv(fixtures_path)

# 3. Reshape Fixtures to Team-Centric Long Format
home_side = df_fixtures[[
    'home_team', 'away_team', 'home_rating', 'away_rating', 'home_winning_expectancy'
]].copy().rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'home_rating': 'team_elo',
    'away_rating': 'opp_elo',
    'home_winning_expectancy': 'win_probability'
})

away_side = df_fixtures[[
    'away_team', 'home_team', 'away_rating', 'home_rating', 'away_winning_expectancy'
]].copy().rename(columns={
    'away_team': 'team',
    'home_team': 'opponent',
    'away_rating': 'team_elo',
    'home_rating': 'opp_elo',
    'away_winning_expectancy': 'win_probability'
})

df_context = pd.concat([home_side, away_side], ignore_index=True)

# 4. Clean and force win_probability into a Float
df_context['win_probability'] = df_context['win_probability'].astype(str).str.replace('%', '', regex=False).str.strip()
df_context['win_probability'] = pd.to_numeric(df_context['win_probability'], errors='coerce')

if df_context['win_probability'].max() > 1.0:
    df_context['win_probability'] = df_context['win_probability'] / 100.0

df_context['win_probability'] = df_context['win_probability'].fillna(0.50)

# Correct the win probability calculation for the away team entries
half_len = len(home_side)
df_context.loc[half_len:, 'win_probability'] = 1.0 - df_context.loc[half_len:, 'win_probability']

df_context['strength_diff'] = df_context['team_elo'] - df_context['opp_elo']
df_context.to_csv(output_fixture_path, index=False)
print(f"✅ Generated and saved fixture context mapping to: {output_fixture_path.name}")

print("\n" + "=" * 60)
print("⚙️ STEP 2: FIXTURE DIFFICULTY ADJUSTMENT FUNCTION")
print("=" * 60)

# 5. Merge Player Baseline Predictions with Fixture Context
df_merged = pd.merge(df_players, df_context, left_on='country', right_on='team', how='inner')

# 6. Define the Tactical Adjustment Function
# 6. Define the Tactical Adjustment Function
def calculate_match_multiplier(row):
    win_prob = row['win_probability']
    # Aggressive scaling: Min 0.50x (0% win prob), Max 1.50x (100% win prob)
    multiplier = 0.50 + (win_prob * 1.0)
    return round(multiplier, 3)

df_merged['fixture_multiplier'] = df_merged.apply(calculate_match_multiplier, axis=1)

# Rename baseline prediction before math to keep it clean
df_merged.rename(columns={'predicted_fantasy_points': 'base_projection'}, inplace=True)
df_merged['match_adjusted_projection'] = (df_merged['base_projection'] * df_merged['fixture_multiplier'])

# ---------------------------------------------------------
# 7. THE FIX: AGGREGATE BACK TO 1 ROW PER PLAYER
# ---------------------------------------------------------
print("Aggregating multi-game projections into a single Group Stage average...")

# We group by all the static player metadata, taking the mean of their adjusted projections
df_final = df_merged.groupby([
    'name', 'country', 'position', 'market_value_in_eur', 'std_fp_last_5', 'base_projection'
]).agg(
    avg_fixture_multiplier=('fixture_multiplier', 'mean'),
    adjusted_projection=('match_adjusted_projection', 'mean'),
    opponents=('opponent', lambda x: ' | '.join(list(x)))  # Strings opponents together cleanly
).reset_index()

# Round the final aggregated values
df_final['adjusted_projection'] = df_final['adjusted_projection'].round(2)
df_final['avg_fixture_multiplier'] = df_final['avg_fixture_multiplier'].round(3)



print("\n" + "=" * 60)
print("🏆 STEP 3: UPDATED TOURNAMENT RECOMMENDATIONS")
print("=" * 60)

# Save full output report
df_final.to_csv(output_final_path, index=False)

# --- RE-RANKING DASHBOARD OUTPUTS ---

print("\n🥇 TOP 5 PROJECTED PLAYERS (Group Stage Avg)")
print("-" * 60)
top_overall = df_final.sort_values(by='adjusted_projection', ascending=False).head(5)
display(top_overall[['name', 'country', 'opponents', 'base_projection', 'adjusted_projection']])

print("\n🛡️ BEST SAFE CAPTAINS (Group Stage Avg)")
print("-" * 60)
df_final['safe_score_adj'] = df_final['adjusted_projection'] / (1 + df_final['std_fp_last_5'] + 0.01)
safe_caps = df_final.sort_values(by='safe_score_adj', ascending=False).head(5)
display(safe_caps[['name', 'country', 'opponents', 'adjusted_projection', 'safe_score_adj']])

print("\n🚀 BEST DIFFERENTIAL CAPTAINS (Group Stage Avg)")
print("-" * 60)
df_final['diff_score_adj'] = df_final['adjusted_projection'] * (1 + df_final['std_fp_last_5'])
diff_caps = df_final.sort_values(by='diff_score_adj', ascending=False).head(5)
display(diff_caps[['name', 'country', 'opponents', 'adjusted_projection', 'diff_score_adj']])

print("\n💎 TOP 5 HIDDEN GEMS (Group Stage Avg)")
print("-" * 60)
df_final['pred_pct_adj'] = df_final['adjusted_projection'].rank(pct=True) * 100
df_final['val_pct'] = df_final['market_value_in_eur'].rank(pct=True) * 100
df_final['gem_score_adj'] = df_final['pred_pct_adj'] - df_final['val_pct']

df_playable_adj = df_final[df_final['pred_pct_adj'] >= 60].copy()
gems = df_playable_adj.sort_values(by='gem_score_adj', ascending=False).head(5)
gems_display = gems[['name', 'country', 'opponents', 'market_value_in_eur', 'adjusted_projection', 'gem_score_adj']].copy()
gems_display['market_value_in_eur'] = gems_display['market_value_in_eur'].map(lambda x: f"€{x:,.0f}")
display(gems_display)

🏗️ STEP 1: GENERATING FIXTURE CONTEXT
✅ Generated and saved fixture context mapping to: fixture_context.csv

⚙️ STEP 2: FIXTURE DIFFICULTY ADJUSTMENT FUNCTION
Aggregating multi-game projections into a single Group Stage average...

🏆 STEP 3: UPDATED TOURNAMENT RECOMMENDATIONS

🥇 TOP 5 PROJECTED PLAYERS (Group Stage Avg)
------------------------------------------------------------


,name,country,opponents,base_projection,adjusted_projection
415,Kylian MBAPPE,France,Senegal | Iraq | Norway,6.69,8.65
607,RAPHINHA,Brazil,Morocco | Haiti | Scotland,6.45,8.43
214,Erling HAALAND,Norway,Senegal | Iraq | France,8.31,8.09
708,VINICIUS JUNIOR,Brazil,Morocco | Haiti | Scotland,5.75,7.51
592,Patrik Schick,Czechia,South Korea | South Africa | Mexico,5.00,5.88



🛡️ BEST SAFE CAPTAINS (Group Stage Avg)
------------------------------------------------------------


,name,country,opponents,adjusted_projection,safe_score_adj
155,Darwin NUNEZ,Uruguay,Saudi Arabia | Cape Verde | Spain,5.20,5.148515
291,Iliman NDIAYE,Senegal,France | Norway | Iraq,3.41,3.376238
275,Hans VANAKEN,Belgium,Egypt | Iran | New Zealand,3.10,3.069307
126,Charles DE KETELAERE,Belgium,Egypt | Iran | New Zealand,2.81,2.782178
693,Thiago ALMADA,Argentina,Austria | Jordan | Algeria,2.67,2.643564



🚀 BEST DIFFERENTIAL CAPTAINS (Group Stage Avg)
------------------------------------------------------------


,name,country,opponents,adjusted_projection,diff_score_adj
607,RAPHINHA,Brazil,Morocco | Haiti | Scotland,8.43,92.603455
592,Patrik Schick,Czechia,South Korea | South Africa | Mexico,5.88,44.212910
217,Esmir BAJRAKTAREVIC,Bosnia and Herzegovina,Switzerland | Qatar | Canada,3.84,37.797413
500,Mats WIEFFER,Netherlands,Sweden | Tunisia | Japan,3.58,36.933613
708,VINICIUS JUNIOR,Brazil,Morocco | Haiti | Scotland,7.51,36.303775



💎 TOP 5 HIDDEN GEMS (Group Stage Avg)
------------------------------------------------------------


,name,country,opponents,market_value_in_eur,adjusted_projection,gem_score_adj
97,Ben WAINE,New Zealand,Iran | Egypt | Belgium,"€50,000",4.86,92.389853
608,RAYAN,Brazil,Morocco | Haiti | Scotland,"€50,000",4.36,88.384513
192,ENDRICK,Brazil,Morocco | Haiti | Scotland,"€50,000",4.34,88.117490
389,Keisuke GOTO,Japan,Netherlands | Tunisia | Sweden,"€50,000",4.07,84.913218
628,Richard RIOS,Colombia,Uzbekistan | DR Congo | Portugal,"€50,000",3.82,81.575434
